In [29]:
import requests
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import urllib3
import time


In [30]:


urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

def pobierz_historie_pse(max_dni=30):
    """
    Pobiera dane z PSE przechodząc po stronach (nextLink).
    max_dni: ile dni wstecz/w przód chcesz pobrać (np. 30, 90, 365)
    """
    url = "https://api.raporty.pse.pl/api/his-wlk-cal"
    headers = {"User-Agent": "Mozilla/5.0"}
    
    wszystkie_rekordy = []
    pobrane_strony = 0
    
    print(f"Pobieranie danych z PSE (maksymalnie {max_dni} dni)...")
    
    while url and pobrane_strony < max_dni:
        response = requests.get(url, headers=headers, verify=False)
        
        if response.status_code != 200:
            print(f"Błąd HTTP {response.status_code}")
            break
            
        data = response.json()
        records = data.get("value", [])
        wszystkie_rekordy.extend(records)
        
        # Pobieramy link do kolejnej strony danych
        url = data.get("nextLink")
        pobrane_strony += 1
        
        if pobrane_strony % 5 == 0:
            print(f"Pobrano {pobrane_strony} dni ({len(wszystkie_rekordy)} wierszy)...")
            
        time.sleep(0.1)  # Krótka pauza, by nie przeciążać serwera PSE
        
    print(f"Zakończono! Łącznie pobrano {len(wszystkie_rekordy)} rekordów.")
    
    # Tworzenie i czyszczenie DataFrame
    df = pd.DataFrame(wszystkie_rekordy)
    wybrane_kolumny = ["dtime", "demand", "pv", "wi", "jg"]
    df = df[wybrane_kolumny].copy()
    
    df["dtime"] = pd.to_datetime(df["dtime"])
    for col in ["demand", "pv", "wi", "jg"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")
        
    df = df.sort_values("dtime").reset_index(drop=True)
    
    # Inżynieria cech
    df["oze_total"] = df["pv"] + df["wi"]
    df["net_demand"] = df["demand"] - df["oze_total"]
    
    return df

# Pobranie np. 30 dni danych na próbę i zapis do pliku
df_energy = pobierz_historie_pse(max_dni=30)
df_energy.to_csv("dane_energetyczne_pse.csv", index=False)
print("Dane zapisane do pliku 'dane_energetyczne_pse.csv'!")

Pobieranie danych z PSE (maksymalnie 30 dni)...
Pobrano 5 dni (500 wierszy)...
Pobrano 10 dni (1000 wierszy)...
Pobrano 15 dni (1500 wierszy)...
Pobrano 20 dni (2000 wierszy)...
Pobrano 25 dni (2500 wierszy)...
Pobrano 30 dni (3000 wierszy)...
Zakończono! Łącznie pobrano 3000 rekordów.
Dane zapisane do pliku 'dane_energetyczne_pse.csv'!


In [31]:
# demand – Rzeczywiste krajowe zapotrzebowanie na moc [MW].
# pv – Generacja z farm fotowoltaicznych [MW].
# wi – Generacja z farm wiatrowych [MW].
# jg – Generacja jednostek konwencjonalnych [MW].
# net_demand - zapotrzebowanie krajowe pomniejoszone od  pv i wi czyli oze (odnawialnych zrodel energi)
# dtime – Dokładny timestamp.

In [32]:
df_energy.head()

,dtime,demand,pv,wi,jg,oze_total,net_demand
0,2024-06-14 00:00:00,15498.469,0.0,296.576,11946.300,296.576,15201.893
1,2024-06-14 00:15:00,16295.296,0.0,299.813,12132.160,299.813,15995.483
2,2024-06-14 00:30:00,15964.026,0.0,320.835,11786.202,320.835,15643.191
3,2024-06-14 00:45:00,15772.470,0.0,341.506,11517.423,341.506,15430.964
4,2024-06-14 01:00:00,15695.535,0.0,349.510,11421.267,349.510,15346.025


In [33]:
df_energy.describe()

,dtime,demand,pv,wi,jg,oze_total,net_demand
count,3000,3000.000000,3000.000000,3000.000000,3000.000000,3000.000000,3000.000000
mean,2024-06-29 14:52:30,17691.331438,3312.248953,1649.527284,9612.053635,4961.776237,12729.555201
min,2024-06-14 00:00:00,11807.507000,0.000000,158.406000,5254.066000,232.334000,4483.653000
25%,2024-06-21 19:26:15,15365.491500,0.000000,584.737000,8189.415500,2131.732250,10754.780500
50%,2024-06-29 14:52:30,17738.476500,1838.900500,1370.091500,9536.163500,3988.184000,12684.846500
75%,2024-07-07 10:18:45,20095.060750,6375.200250,2477.416250,10929.775500,7988.256250,14924.590000
max,2024-07-15 05:45:00,22609.531000,11582.160000,5507.236000,15951.763000,14496.033000,19974.903000
std,NaN,2653.778461,3477.725043,1200.972824,1987.639783,3415.194546,3265.999469


In [34]:
df_energy.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3000 entries, 0 to 2999
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   dtime       3000 non-null   datetime64[ns]
 1   demand      3000 non-null   float64       
 2   pv          3000 non-null   float64       
 3   wi          3000 non-null   float64       
 4   jg          3000 non-null   float64       
 5   oze_total   3000 non-null   float64       
 6   net_demand  3000 non-null   float64       
dtypes: datetime64[ns](1), float64(6)
memory usage: 164.2 KB
